# Notebook 02 — Pratique du nettoyage SQL + Python
**Base :** `cyclistic_db` | **Table source :** `cyclistic_raw`  
**Objectif :** Reproduire le pipeline de nettoyage depuis les données brutes,  
en SQL pur d'abord, puis en Python (pandas), pour comprendre les deux approches.

---
## Plan
1. Connexion & exploration de `cyclistic_raw`
2. Étape 1 — Comprendre le schéma brut et ses problèmes
3. Étape 2 — Conversion des types (TEXT → TIMESTAMP, NUMERIC)
4. Étape 3 — Calcul de `ride_length` en SQL
5. Étape 4 — Extraction des variables temporelles
6. Étape 5 — Détection et suppression des valeurs invalides
7. Étape 6 — Suppression des doublons
8. Étape 7 — Filtrage des stations de maintenance
9. Étape 8 — Créer la table nettoyée `cyclistic_clean_v2`
10. Validation : comparer avec `cyclistic_clean` officielle
11. Approche Python (pandas) — le même pipeline
12. Exercices


---
## 1. Connexion & exploration

In [ ]:
import psycopg2
import pandas as pd

conn = psycopg2.connect(
    host="localhost", port=5432,
    dbname="cyclistic_db",
    user="postgres", password="postgres"
)

def run(sql, conn=conn):
    cur = conn.cursor()
    cur.execute(sql)
    rows = cur.fetchall()
    cols = [d[0] for d in cur.description]
    cur.close()
    return pd.DataFrame(rows, columns=cols)

def execute(sql, conn=conn):
    """Exécute une requête sans retour de données (CREATE, DROP, INSERT...)"""
    cur = conn.cursor()
    cur.execute(sql)
    conn.commit()
    cur.close()
    print("OK")

print("Connecté.")

---
## 2. Explorer le schéma brut

Dans `cyclistic_raw`, **toutes les colonnes sont en TEXT** — c'est intentionnel.  
C'est l'état réel d'un fichier CSV importé sans nettoyage préalable.  
Notre travail : identifier les problèmes et les corriger.

In [ ]:
# Structure de la table brute
schema = run("""
SELECT column_name, data_type
FROM information_schema.columns
WHERE table_name = 'cyclistic_raw'
ORDER BY ordinal_position;
""")
print("Schéma de cyclistic_raw :")
schema

In [ ]:
# Nombre total de lignes brutes
run("SELECT COUNT(*) AS total_brut FROM cyclistic_raw;")

In [ ]:
# Aperçu des 5 premières lignes
run("SELECT * FROM cyclistic_raw LIMIT 5;")

In [ ]:
# Répartition par fichier source
run("""
SELECT source_file, COUNT(*) AS lignes
FROM cyclistic_raw
GROUP BY source_file
ORDER BY source_file;
""")

---
## 3. Identifier les problèmes

Avant de nettoyer, on **diagnostique** : combien de valeurs manquantes, invalides, doublons ?

In [ ]:
# Valeurs nulles par colonne
run("""
SELECT
    SUM(CASE WHEN ride_id            IS NULL OR ride_id = ''            THEN 1 ELSE 0 END) AS ride_id_null,
    SUM(CASE WHEN started_at         IS NULL OR started_at = ''         THEN 1 ELSE 0 END) AS started_at_null,
    SUM(CASE WHEN ended_at           IS NULL OR ended_at = ''           THEN 1 ELSE 0 END) AS ended_at_null,
    SUM(CASE WHEN start_station_name IS NULL OR start_station_name = '' THEN 1 ELSE 0 END) AS start_station_null,
    SUM(CASE WHEN end_station_name   IS NULL OR end_station_name = ''   THEN 1 ELSE 0 END) AS end_station_null,
    SUM(CASE WHEN member_casual      IS NULL OR member_casual = ''      THEN 1 ELSE 0 END) AS member_null
FROM cyclistic_raw;
""")

In [ ]:
# Valeurs distinctes de member_casual (vérifier les anomalies)
run("""
SELECT member_casual, COUNT(*) AS nb
FROM cyclistic_raw
GROUP BY member_casual
ORDER BY nb DESC;
""")

In [ ]:
# Doublons sur ride_id
run("""
SELECT COUNT(*) AS total, COUNT(DISTINCT ride_id) AS uniques,
       COUNT(*) - COUNT(DISTINCT ride_id) AS doublons
FROM cyclistic_raw;
""")

In [ ]:
# Stations de maintenance (à supprimer)
run("""
SELECT start_station_name, COUNT(*) AS nb
FROM cyclistic_raw
WHERE start_station_name ~* 'HQ QR|DIVVY|TEST|Hubbard Bike-checking'
GROUP BY start_station_name
ORDER BY nb DESC;
""")

---
## 4. Nettoyage en SQL — Vue intermédiaire

On crée une **vue SQL** (`VIEW`) qui applique toutes les transformations à la volée.  
Une vue est une requête enregistrée — elle ne stocke pas de données, elle les calcule à la demande.

**Transformations :**
- Conversion `started_at`, `ended_at` → TIMESTAMP
- Calcul `ride_length` en minutes
- Extraction `day_of_week`, `month`, `year`, `hour_of_day`, `season`
- Filtrage des lignes invalides
- `~*` = opérateur PostgreSQL pour les expressions régulières (case-insensitive)

In [ ]:
execute("""
DROP VIEW IF EXISTS v_cyclistic_clean;
""")

execute("""
CREATE VIEW v_cyclistic_clean AS
SELECT
    ride_id,
    rideable_type,

    -- Étape 1 : convertir TEXT → TIMESTAMP
    started_at::TIMESTAMP                                        AS started_at,
    ended_at::TIMESTAMP                                          AS ended_at,

    -- Étape 2 : calculer la durée en minutes
    ROUND(
        EXTRACT(EPOCH FROM (ended_at::TIMESTAMP - started_at::TIMESTAMP)) / 60.0
    ::numeric, 2)                                                AS ride_length,

    -- Étape 3 : extraire les variables temporelles
    -- day_of_week : 1=Dimanche, 7=Samedi (convention du projet)
    CASE EXTRACT(DOW FROM started_at::TIMESTAMP)::INT
        WHEN 0 THEN 1   -- Dimanche
        WHEN 1 THEN 2   -- Lundi
        WHEN 2 THEN 3
        WHEN 3 THEN 4
        WHEN 4 THEN 5
        WHEN 5 THEN 6
        WHEN 6 THEN 7   -- Samedi
    END                                                          AS day_of_week,

    EXTRACT(MONTH FROM started_at::TIMESTAMP)::SMALLINT          AS month,
    EXTRACT(YEAR  FROM started_at::TIMESTAMP)::SMALLINT          AS year,
    EXTRACT(HOUR  FROM started_at::TIMESTAMP)::SMALLINT          AS hour_of_day,

    -- Étape 4 : saison
    CASE EXTRACT(MONTH FROM started_at::TIMESTAMP)::INT
        WHEN 12 THEN 'Winter' WHEN 1 THEN 'Winter' WHEN 2 THEN 'Winter'
        WHEN  3 THEN 'Spring' WHEN 4 THEN 'Spring' WHEN 5 THEN 'Spring'
        WHEN  6 THEN 'Summer' WHEN 7 THEN 'Summer' WHEN 8 THEN 'Summer'
        ELSE 'Fall'
    END                                                          AS season,

    start_station_name,
    start_station_id,
    end_station_name,
    end_station_id,
    NULLIF(start_lat, '')::NUMERIC(10,6)                         AS start_lat,
    NULLIF(start_lng, '')::NUMERIC(10,6)                         AS start_lng,
    NULLIF(end_lat,   '')::NUMERIC(10,6)                         AS end_lat,
    NULLIF(end_lng,   '')::NUMERIC(10,6)                         AS end_lng,
    member_casual

FROM (
    -- Sous-requête : dédoublonner sur ride_id
    SELECT DISTINCT ON (ride_id) *
    FROM cyclistic_raw
    ORDER BY ride_id, started_at
) deduped

WHERE
    -- Étape 5 : supprimer les durées invalides
    EXTRACT(EPOCH FROM (ended_at::TIMESTAMP - started_at::TIMESTAMP)) / 60.0 > 0
    AND
    EXTRACT(EPOCH FROM (ended_at::TIMESTAMP - started_at::TIMESTAMP)) / 60.0 <= 1440
    -- Étape 6 : supprimer les stations de maintenance
    AND (start_station_name !~* 'HQ QR|DIVVY|TEST|Hubbard Bike-checking'
         OR start_station_name IS NULL)
    AND (end_station_name !~* 'HQ QR|DIVVY|TEST|Hubbard Bike-checking'
         OR end_station_name IS NULL);
""")
print("Vue v_cyclistic_clean créée.")

---
## 5. Vérifier la vue

In [ ]:
# Compter les lignes après nettoyage
run("SELECT COUNT(*) AS lignes_nettoyees FROM v_cyclistic_clean;")

In [ ]:
# Aperçu des premières lignes nettoyées
run("SELECT * FROM v_cyclistic_clean LIMIT 5;")

In [ ]:
# Statistiques descriptives depuis la vue nettoyée
run("""
SELECT
    member_casual,
    COUNT(*)                                    AS total_rides,
    ROUND(AVG(ride_length)::numeric, 2)         AS avg_min,
    ROUND(MIN(ride_length)::numeric, 2)         AS min_min,
    ROUND(MAX(ride_length)::numeric, 2)         AS max_min
FROM v_cyclistic_clean
GROUP BY member_casual;
""")

---
## 6. Matérialiser la table nettoyée `cyclistic_clean_v2`

La vue recalcule tout à chaque requête (lent sur 3,9M lignes).  
On la **matérialise** en une vraie table pour les analyses rapides.

In [ ]:
execute("DROP TABLE IF EXISTS cyclistic_clean_v2;")
execute("""
CREATE TABLE cyclistic_clean_v2 AS
SELECT * FROM v_cyclistic_clean;
""")
run("SELECT COUNT(*) AS total FROM cyclistic_clean_v2;")

---
## 7. Validation : comparer avec la table officielle

On compare `cyclistic_clean_v2` (nettoyée en SQL) avec `cyclistic_clean` (nettoyée en Python).  
Les résultats doivent être très proches — de petites différences peuvent exister selon les décisions de nettoyage.

In [ ]:
run("""
SELECT
    'cyclistic_clean (Python)' AS source,
    COUNT(*) AS total_rides,
    ROUND(AVG(ride_length)::numeric, 2) AS avg_min
FROM cyclistic_clean

UNION ALL

SELECT
    'cyclistic_clean_v2 (SQL)' AS source,
    COUNT(*) AS total_rides,
    ROUND(AVG(ride_length)::numeric, 2) AS avg_min
FROM cyclistic_clean_v2;
""")

---
## 8. Le même pipeline en Python (pandas)

Voici comment reproduire exactement les mêmes étapes depuis la base PostgreSQL,  
mais en utilisant **pandas** à la place du SQL.

In [ ]:
import numpy as np

# Charger le brut depuis PostgreSQL dans un DataFrame
print("Chargement cyclistic_raw depuis PostgreSQL...")
cur = conn.cursor()
cur.execute("SELECT * FROM cyclistic_raw")
rows = cur.fetchall()
cols = [d[0] for d in cur.description]
cur.close()

df = pd.DataFrame(rows, columns=cols)
print(f"Lignes brutes chargées : {len(df):,}")
df.head(3)

In [ ]:
# Étape 1 : convertir les dates
df['started_at'] = pd.to_datetime(df['started_at'], infer_datetime_format=True, errors='coerce')
df['ended_at']   = pd.to_datetime(df['ended_at'],   infer_datetime_format=True, errors='coerce')
print(f"Dates converties. NaT started_at : {df['started_at'].isna().sum():,}")

In [ ]:
# Étape 2 : calculer ride_length en minutes
df['ride_length'] = (df['ended_at'] - df['started_at']).dt.total_seconds() / 60
df['ride_length'] = df['ride_length'].round(2)
print(f"ride_length calculé. Exemples : {df['ride_length'].head().tolist()}")

In [ ]:
# Étape 3 : variables temporelles
DOW_MAP = {6: 1, 0: 2, 1: 3, 2: 4, 3: 5, 4: 6, 5: 7}  # pandas: lundi=0 → notre convention
df['day_of_week'] = df['started_at'].dt.dayofweek.map(DOW_MAP)
df['month']       = df['started_at'].dt.month
df['year']        = df['started_at'].dt.year
df['hour_of_day'] = df['started_at'].dt.hour
df['season']      = df['month'].map({
    12:'Winter', 1:'Winter', 2:'Winter',
     3:'Spring', 4:'Spring', 5:'Spring',
     6:'Summer', 7:'Summer', 8:'Summer',
     9:'Fall',  10:'Fall',  11:'Fall'
})
print("Variables temporelles créées.")
df[['started_at','day_of_week','month','year','hour_of_day','season']].head(3)

In [ ]:
# Étape 4 : supprimer les doublons
avant = len(df)
df = df.drop_duplicates(subset=['ride_id'])
print(f"Doublons supprimés : {avant - len(df):,}  |  Reste : {len(df):,}")

In [ ]:
# Étape 5 : supprimer les durées invalides
avant = len(df)
df = df[(df['ride_length'] > 0) & (df['ride_length'] <= 1440)]
print(f"Durées invalides supprimées : {avant - len(df):,}  |  Reste : {len(df):,}")

In [ ]:
# Étape 6 : supprimer les stations de maintenance
avant = len(df)
pattern = 'HQ QR|DIVVY|TEST|Hubbard Bike-checking'
mask = (
    df['start_station_name'].str.contains(pattern, case=False, na=False) |
    df['end_station_name'].str.contains(pattern, case=False, na=False)
)
df = df[~mask]
print(f"Stations maintenance supprimées : {avant - len(df):,}  |  Reste : {len(df):,}")

In [ ]:
# Résultat final
print(f"\n=== RÉSULTAT FINAL ===")
print(f"Lignes brutes  : 3,906,752")
print(f"Lignes nettoyées (Python) : {len(df):,}")
print(f"\nStatistiques :")
print(df.groupby('member_casual')['ride_length'].agg(['count','mean','median']).round(2))

---
## Exercices

### Exercice 1 — SQL
Dans `cyclistic_raw`, certaines lignes ont `started_at > ended_at` (heure de fin avant heure de début).  
Combien y en a-t-il ? Écris la requête SQL pour les compter.

In [ ]:
# Ton code ici
ex1 = """
-- Compte les lignes où la fin est avant le début

"""
# run(ex1)

### Exercice 2 — SQL
Calcule le **pourcentage de valeurs nulles** pour `start_station_name`  
séparément pour les fichiers 2019 et les fichiers 2020.

In [ ]:
# Ton code ici
ex2 = """
-- Indice : utilise source_file LIKE '2019%' pour distinguer

"""
# run(ex2)

### Exercice 3 — Python
Depuis le DataFrame `df` nettoyé, ajoute une colonne `is_weekend`  
qui vaut `1` si le trajet a eu lieu un samedi ou dimanche, `0` sinon.

In [ ]:
# Ton code ici
# Indice : day_of_week == 1 (Dimanche) ou == 7 (Samedi) dans notre convention

# df['is_weekend'] = ...

In [ ]:
conn.close()
print("Connexion fermée.")